In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as ticker


def _draw_profile_frame(fig, civil, gap_in=0.30, band_in=0.30, margin_in=0.15,
                        color='#2C5F8A', hatch='///'):
    """Encadre la page d'un liere hachure et pose la pastille jaune CIVIL.

    Marqueur visuel : une planche civile (para 80 kg, pilote 86 kg) ne doit pas
    pouvoir etre confondue avec la planche militaire (90 kg / 80 kg) du meme
    avion une fois imprimee.

    Retourne la valeur a passer a `pdf.savefig(bbox_inches=...)` : 'tight' si
    le cadre est desactive, sinon la bbox qui englobe le cadre.
    """
    if not civil:
        return 'tight'

    import matplotlib as _mpl
    from matplotlib.transforms import Bbox as _Bbox
    from matplotlib.patches import Rectangle as _Rect

    # bbox du contenu, mesuree avant d'ajouter le cadre
    bb = fig.get_tightbbox()
    x0 = bb.x0 - gap_in - band_in
    y0 = bb.y0 - gap_in - band_in
    x1 = bb.x1 + gap_in + band_in
    y1 = bb.y1 + gap_in + band_in
    w_in, h_in = fig.get_size_inches()

    def _rect(rx, ry, rw, rh, **kw):
        r = _Rect((rx / w_in, ry / h_in), rw / w_in, rh / h_in,
                  transform=fig.transFigure, clip_on=False, **kw)
        fig.add_artist(r)
        return r

    # les 4 bandes du liere (hatch.linewidth est fige a la construction)
    with _mpl.rc_context({'hatch.linewidth': 2.0}):
        band_kw = dict(facecolor='white', edgecolor=color, hatch=hatch,
                       linewidth=1.2, zorder=1000)
        _rect(x0, y0, x1 - x0, band_in, **band_kw)
        _rect(x0, y1 - band_in, x1 - x0, band_in, **band_kw)
        _rect(x0, y0 + band_in, band_in, (y1 - y0) - 2 * band_in, **band_kw)
        _rect(x1 - band_in, y0 + band_in, band_in, (y1 - y0) - 2 * band_in,
              **band_kw)

    # pastille CIVIL, a cheval sur la bande du haut
    badge_w, badge_h = 2.0, 0.44
    bx = (x0 + x1) / 2 - badge_w / 2
    by = y1 - band_in / 2 - badge_h / 2
    _rect(bx, by, badge_w, badge_h, facecolor='#FFD24D', edgecolor='#B8860B',
          linewidth=1.5, zorder=1001)
    fig.text((bx + badge_w / 2) / w_in, (by + badge_h / 2) / h_in, 'CIVIL',
             ha='center', va='center', fontsize=22, fontweight='bold',
             color='#3A2E00', zorder=1002)

    m = margin_in
    return _Bbox([[x0 - m, y0 - m], [x1 + m, y1 + m]])


def plot_conf(config_, x_slots_top, x_slots_middle, x_slots_bot, zones, ax, extra_slots, sk_limit, EW_LBS=None, EW_MOMENT=None, VERSION=None):

    # Add zones to the plot
    for i, zone in enumerate(zones):
        if i < 6:
            rect = patches.Polygon(xy=[(zone["x0"], -zone["height"]/2),
                                        (zone["x0"], zone["height"]/2),
                                        (zone["x1"], zones[i+1]["height"]/2),
                                        (zone["x1"], -zones[i+1]["height"]/2)], lw=4, fill=False)
        if i == 6:
            rect = patches.Polygon(xy=[(zone["x0"], -zone["height"]/2),
                                        (zone["x0"], zone["height"]/2),
                                        (zone["x1"], zone["height"]/2 - 5),
                                        (zone["x1"], -zone["height"]/2 +5)], lw=4, fill=False)

        ax.add_patch(rect)
        ax.text(zone["arm"], 0, zone["label"],
                horizontalalignment='center', verticalalignment='center')

    # Porte
    ax.add_patch(patches.Polygon(xy=[(zones[4]["x0"], -zones[4]["height"]/2+2),
                                    (zones[4]["x0"], -zones[4]["height"]/2),
                                    (zones[6]["x0"], -zones[6]["height"]/2),
                                    (zones[6]["x0"], -zones[6]["height"]/2 +2)], lw=4, fill=False))
        # Aile
    ax.add_patch(patches.Polygon(xy=[(177.57, zones[1]["height"]/2 + 1),
                                    (177.57, zones[1]["height"]/2 + 30),
                                    (177.57+66.40, zones[3]["height"]/2 + 20),
                                    (177.57+66.40, zones[3]["height"]/2)], lw=4, fill=False))

        # Aile
    ax.add_patch(patches.Polygon(xy=[(177.57, -zones[1]["height"]/2 - 1),
                                    (177.57, -zones[1]["height"]/2 - 30),
                                    (177.57+66.40, -zones[3]["height"]/2 - 20),
                                    (177.57+66.40, -zones[3]["height"]/2)], lw=4, fill=False))


        # roue
    ax.add_patch(patches.Polygon(xy=[(177.57, -zones[1]["height"]/2 - 1),
                                    (177.57, -zones[1]["height"]/2 - 30),
                                    (177.57+66.40, -zones[3]["height"]/2 - 20),
                                    (177.57+66.40, -zones[3]["height"]/2)], lw=4, fill=False))

        # roue
    ax.add_patch(patches.Polygon(xy=[(228, 70+5),
                                    (200, 70+5),
                                    (200, 70-5),
                                    (228,70-5) ], lw=4, fill=False))



    ax.add_patch(patches.Polygon(xy=[(228, -70+5),
                                    (200, -70+5),
                                    (200, -70-5),
                                    (228,-70-5) ], lw=4, fill=False))
    ax.add_patch(patches.Polygon(xy=[(250, zones[3]["height"]/2),
                                    (250, -zones[3]["height"]/2), ], lw=4, fill=False, color='red', linestyle='--'))
    ax.add_patch(patches.Polygon(xy=[(204, zones[3]["height"]/2),
                                    (204, -zones[3]["height"]/2), ], lw=4, fill=False, linestyle='--'))

    ax.add_patch(rect)

    ax.add_patch(patches.Circle(((zones[0]["x1"]- zones[0]["x0"])/2 + zones[0]["x0"], -zones[0]["height"]/4), 10, fill=True, color='blue'))

    y_slots = [zones[3]["height"] / 4] * len(x_slots_top) + [0] * len(x_slots_middle) + [-zones[3]["height"] / 4] * len(x_slots_bot)
    x_slots = x_slots_top + x_slots_middle + x_slots_bot


    for i in range(1, sk_limit+1):
        x = x_slots[config_.index(i)]
        y = y_slots[config_.index(i)]
        ax.add_patch(patches.Circle((x,  y), 9, fill=True, color='red'))
        ax.text(x, y, str(i), horizontalalignment='center', verticalalignment='center',  size=30)


    ax.set_ylim(-55, 55)
    ax.set_xlim(70, 400)

    ax.axis('on')
    ax.axis('equal')


    current_num_xticks = len(ax.get_xticks())
    current_num_yticks = len(ax.get_yticks())

    # Set new locator for x and y axes to double the number of ticks
    # The bin parameter helps in controlling the maximum number of ticks
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=current_num_xticks * 4))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=current_num_yticks * 4))

    # Making ticks bigger and setting labels in French
    ax.tick_params(axis='both', which='major', labelsize=14, width=2, length=6)
    ax.tick_params(axis='both', which='minor', labelsize=10, width=1, length=4)


    plt.xlabel('Longueur en inches', fontsize=20)
    plt.ylabel('Largeur en inches', fontsize=20)
    placements_str = ', '.join([str(round(x, 2)) for x in x_slots])
    ew_str = ''
    if EW_LBS is not None and EW_MOMENT is not None:
        ew_str = f'\nEW: {EW_LBS} lbs  |  EW Moment: {EW_MOMENT * 1000:.0f} lbs·in'
    if VERSION is not None:
        ew_str += f'\nVersion: {VERSION}'
    plt.title(f'Positionnement des paras en fonction du nombre à bord\n \
    Bras de levier paras (inches): \n {placements_str}{ew_str}', fontsize=25)

    # Adding grid, labels, title, and adjusting tick size
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)

# --------------------------------------------------------------------------
# Placement dynamique (pas de sieges fixes, les paras sont repartis dans la
# soute selon leur nombre pour viser un CG dans la zone seulement limitee
# par la MTOW : 32.5 - 40 %MAC pour le Caravan).
# --------------------------------------------------------------------------

def _row_counts_caravan(n):
    """Repartit n paras Caravan : copilote (>=10) + 3 rangees cargo.
    Rangee du milieu progressive : 0 si n<12, sinon (n-10)//2.
    Retourne (top, mid, bot, copilot)."""
    copilot = 1 if n >= 10 else 0
    n_cargo = n - copilot
    mid = 0 if n < 12 else max(0, (n - 10) // 2)
    side = n_cargo - mid
    top = (side + 1) // 2
    bot = side // 2
    return top, mid, bot, copilot


def _compute_target_xbar_caravan(n, EW_LBS, EW_MOMENT, MTOW_LBS,
                                  POIDS_PILOTE_KG, POIDS_PARA_KG,
                                  target_cg_in=200.5,
                                  FUEL_TANK_MAX_GAL=332):
    """Bras moyen x̄ pour que le CG a masse-max atteigne target_cg_in (inches)."""
    if n <= 0:
        return None
    LBS2KG = 2.20462
    PARA_LBS = POIDS_PARA_KG * LBS2KG
    PILOT_LBS = POIDS_PILOTE_KG * LBS2KG
    # Max mass for this N : soit tank plein, soit MTOW.
    m_base = EW_LBS + PILOT_LBS + n * PARA_LBS
    # Essai avec tank plein -> si depasse MTOW, reduire le fuel.
    _, m_full, _ = compute_moment([], EW_LBS, EW_MOMENT, POIDS_PARA_KG,
                                    POIDS_PILOTE_KG, FUEL_TANK_MAX_GAL)
    fuel_w_full = m_full - EW_LBS - PILOT_LBS
    if m_base + fuel_w_full <= MTOW_LBS:
        fuel_gal = FUEL_TANK_MAX_GAL
    else:
        # Dichotomie sur fuel_gal pour atteindre MTOW
        lo, hi = 0.0, FUEL_TANK_MAX_GAL
        for _ in range(40):
            mid = (lo + hi) / 2
            _, m_trial, _ = compute_moment([], EW_LBS, EW_MOMENT, POIDS_PARA_KG,
                                            POIDS_PILOTE_KG, mid)
            fuel_w = m_trial - EW_LBS - PILOT_LBS
            if m_base + fuel_w > MTOW_LBS:
                hi = mid
            else:
                lo = mid
        fuel_gal = lo
    # Moment et masse sans paras avec ce fuel
    moment_base, mass_base, _ = compute_moment([], EW_LBS, EW_MOMENT,
                                                 POIDS_PARA_KG, POIDS_PILOTE_KG,
                                                 fuel_gal)
    mass = mass_base + n * PARA_LBS
    target_moment = target_cg_in * mass / 1000
    paras_moment = target_moment - moment_base
    x_bar = paras_moment * 1000 / (n * PARA_LBS)
    return x_bar


def _get_para_positions_caravan(n, zones, EW_LBS, EW_MOMENT, MTOW_LBS,
                                 POIDS_PILOTE_KG, POIDS_PARA_KG,
                                 target_cg_in=200.5,
                                 min_spacing_in=12.0):
    """Retourne [(x, y), ...] : positions des n paras (copilote + 3 rangees cargo).
    Arriere des rangees laterales fixe a x_cargo_max (307, frontiere zones 4/5).
    L'avant est calcule pour preserver x_bar_cargo (centrage a la cible), clampe
    a x_cargo_min si besoin. La rangee du milieu est en quincounce (decale d'une
    demi-interval) par rapport aux laterales.
    Cas special N=20 : on utilise toute la soute (front = x_cargo_min)."""
    if n <= 0:
        return []
    x_cargo_min = zones[1]["x0"]
    x_cargo_max = zones[4]["x1"]
    h = zones[3]["height"]
    y_rows = [h / 4, 0.0, -h / 4]
    copilot_arm = zones[0]["arm"]
    copilot_y = h / 4

    top, mid, bot, copilot = _row_counts_caravan(n)
    counts = [top, mid, bot]
    n_cargo = n - copilot

    positions = []
    if copilot:
        positions.append((copilot_arm, copilot_y))

    if n_cargo == 0:
        return positions

    x_bar = _compute_target_xbar_caravan(
        n, EW_LBS, EW_MOMENT, MTOW_LBS,
        POIDS_PILOTE_KG, POIDS_PARA_KG,
        target_cg_in=target_cg_in)
    if copilot:
        x_bar_cargo = (n * x_bar - copilot_arm) / n_cargo
    else:
        x_bar_cargo = x_bar

    rear = x_cargo_max
    r_side = max(top, bot)
    if n == 20:
        front = x_cargo_min
    elif r_side > 1:
        front = max(x_cargo_min, 2 * x_bar_cargo - rear)
    else:
        front = x_bar_cargo

    s = (rear - front) / (r_side - 1) if r_side > 1 else 0.0

    def _xs_row(r, f, b):
        if r == 0:
            return []
        if r == 1:
            return [(f + b) / 2]
        return [f + i * (b - f) / (r - 1) for i in range(r)]

    xs_top = _xs_row(top, front, rear)
    xs_bot = _xs_row(bot, front, rear)

    if mid > 0:
        mid_front = front + s / 2
        mid_rear = rear - s / 2
        xs_mid = _xs_row(mid, mid_front, mid_rear)
    else:
        xs_mid = []

    for r, xs, y in [(top, xs_top, y_rows[0]),
                     (mid, xs_mid, y_rows[1]),
                     (bot, xs_bot, y_rows[2])]:
        for x in xs:
            positions.append((x, y))
    return positions


def _plot_conf_mini(ax, positions, zones, n_paras, pilot_xy, xlim, ylim, unit='in'):
    """Dessine la configuration : soute + pilote + n paras + labels des bras."""
    for i in range(6):
        if i + 1 < len(zones):
            rect = patches.Polygon(
                xy=[(zones[i]["x0"], -zones[i]["height"]/2),
                    (zones[i]["x0"], zones[i]["height"]/2),
                    (zones[i]["x1"], zones[i+1]["height"]/2),
                    (zones[i]["x1"], -zones[i+1]["height"]/2)],
                lw=1.2, fill=False)
            ax.add_patch(rect)

    px, py = pilot_xy
    ax.add_patch(patches.Circle((px, py), 5.5, fill=True, color='blue', zorder=3))
    ax.text(px, py, 'P', ha='center', va='center', color='white',
            fontweight='bold', size=7, zorder=4)

    for x, y in positions:
        ax.add_patch(patches.Circle((x, y), 5.5, fill=True, color='red', zorder=3))
        # Label du bras AU-DESSUS de chaque para (police plus lisible).
        y_lbl = y + 10
        ax.text(x, y_lbl, f'{x:.0f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color='black', zorder=4)

    if n_paras > 0 and positions:
        bras = sum(x for x, _ in positions) / len(positions)
        title = f'{n_paras} paras  |  bras moyen = {bras:.1f} {unit}'
    else:
        title = '0 para (pilote seul)'
    ax.set_title(title, fontsize=15, fontweight='bold')

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])


def plot_conf_grid(axes_grid, zones, sk_limit, EW_LBS, EW_MOMENT, MTOW_LBS,
                   POIDS_PILOTE_KG, POIDS_PARA_KG, n_values=None,
                   target_cg_in=200.5):
    pilot_xy = ((zones[0]["x1"] - zones[0]["x0"]) / 2 + zones[0]["x0"],
                -zones[0]["height"] / 4)
    xlim = (zones[0]["x0"] - 5, zones[6]["x1"] + 5 if len(zones) > 6 else zones[5]["x1"] + 5)
    ylim = (-45, 45)

    flat_axes = axes_grid.flatten()
    if n_values is None:
        n_values = [0] + list(range(4, sk_limit + 1))
    for i, n in enumerate(n_values):
        positions = _get_para_positions_caravan(
            n, zones, EW_LBS, EW_MOMENT, MTOW_LBS,
            POIDS_PILOTE_KG, POIDS_PARA_KG,
            target_cg_in=target_cg_in)
        _plot_conf_mini(flat_axes[i], positions, zones, n,
                        pilot_xy, xlim, ylim, unit='in')
    for j in range(len(n_values), len(flat_axes)):
        flat_axes[j].axis('off')


In [2]:
import numpy as np

def compute_moment(x_list, EW_LBS, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel):
    moment = EW_MOMENT
    poids = EW_LBS

    POIDS_PILOTE_LBS = 2.20462 * POIDS_PILOTE_KG
    POIDS_PARA_LBS = 2.20462 * POIDS_PARA_KG

    #data gallons, weight, momen
    data = np.array([
 [5, 34, 6.8],
 [10, 67, 13.6],
 [15, 101, 20.4],
 [20, 134, 27.2],
 [25, 168, 34.0],
 [30, 201, 40.8],
 [35, 235, 47.6],
 [40, 268, 54.4],
 [45, 302, 61.2],
 [50, 335, 68.0],
 [55, 369, 74.8],
 [60, 402, 81.6],
 [65, 436, 88.4],
 [70, 469, 95.2],
 [75, 503, 102.0],
 [80, 536, 108.8],
 [85, 570, 115.7],
 [90, 603, 122.5],
 [95, 637, 129.3],
 [100, 670, 136.1],
 [105, 704, 142.9],
 [110, 737, 149.7],
 [115, 771, 156.6],
 [120, 804, 163.4],
 [125, 838, 170.2],
 [130, 871, 177.0],
 [135, 905, 183.8],
 [140, 938, 190.6],
 [145, 972, 197.5],
 [155, 1039, 211.1],
 [160, 1072, 217.9],
 [170, 1139, 231.5],
 [175, 1173, 238.4],
 [180, 1206, 245.2],
 [185, 1240, 252.0],
 [190, 1273, 258.8],
 [195, 1307, 265.7],
 [200, 1340, 272.5],
 [205, 1374, 279.3],
 [210, 1407, 286.1],
 [215, 1441, 292.9],
 [220, 1474, 299.7],
 [225, 1508, 306.5],
 [230, 1541, 313.3],
 [235, 1575, 320.1],
 [240, 1608, 326.9],
 [245, 1642, 333.7],
 [250, 1675, 340.5],
 [255, 1709, 347.3],
 [260, 1742, 354.1],
 [265, 1776, 360.9],
 [270, 1809, 367.7],
 [275, 1843, 374.5],
 [280, 1876, 381.2],
 [285, 1910, 388.0],
 [290, 1943, 394.8],
 [295, 1977, 401.6],
 [300, 2010, 408.4],
 [305, 2044, 415.2],
 [310, 2077, 422.0],
 [320, 2144, 435.6],
 [325, 2178, 442.4],
 [327, 2189, 444.7],
 [330, 2211, 449.1],
 [332, 2224, 451.7]
])

    fuel_moment = np.interp(fuel, data[:, 0], data[:, 2])
    fuel_weight = np.interp(fuel, data[:, 0], data[:, 1])

    moment += fuel_moment
    poids += fuel_weight

    moment += POIDS_PILOTE_LBS * 135.5 / 1000
    poids += POIDS_PILOTE_LBS

    for x in x_list:
        moment += (POIDS_PARA_LBS) * x / 1000
        poids += POIDS_PARA_LBS

    return moment, poids, moment / poids * 1000

In [3]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as ticker  # For controlling tick locations and formats

def plot_envelope(config_, x_slots_top, x_slots_middle, x_slots_bot, EW_LBS, MTOW_LBS, EW_MOMENT, POIDS_PARA_KG,
                  POIDS_PILOTE_KG, ax, list_nb_para_plot, DATUM_LINE, MAC):
    LBS2KG = 2.20462
    EW_KG = EW_LBS / LBS2KG
    MTOW_KG = MTOW_LBS / LBS2KG
    GAL2L = 3.78541
    DENSITY_FUEL_LBS_PER_GAL = 6.7

    def cg_inches_to_mac_percent(cg_inches, datum_line, mac):
        return (cg_inches - datum_line) / mac * 100


    ENVELOPE = [(179.6, EW_LBS),
                (179.6, 5500),
                (193.37, 8000),
                (199.15, MTOW_LBS),
                (204.35, MTOW_LBS),
                (204.35, EW_LBS)]

    droite_oblique = lambda x: (MTOW_LBS - 5500) / (199 - 179.6) * (x - 179.6) + 5500

    rect = patches.Polygon(xy=ENVELOPE, fill=False)
    ax.add_patch(rect)

    x_list = x_slots_top + x_slots_middle + x_slots_bot

    for sk in [0] + list_nb_para_plot:
        if sk == 0:
            list_positions = []
        else:
            list_positions = [x_list[config_.index(i)] for i in range(1, sk + 1)]

        moment, poids, cg = compute_moment(list_positions, EW_LBS, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, 5)

        masse_dep = False
        fuel_de_dep_gal = None
        cg_list, poids_list = [], []
        fuel_range = list(range(0, int(332 * DENSITY_FUEL_LBS_PER_GAL), 100)) + [int(332 * DENSITY_FUEL_LBS_PER_GAL)]

        if sk > 8 or sk == 0:
            ax.text(cg, poids, str(sk), horizontalalignment='right', verticalalignment='center', size=20, color='red')
            cg0, poids0 = cg, poids
            for i, fuel_lbs in enumerate(fuel_range):  # Iterate in steps of 100 lbs
                fuel = fuel_lbs / DENSITY_FUEL_LBS_PER_GAL
                moment, poids, cg = compute_moment(list_positions, EW_LBS, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel)
                cg_list.append(cg)
                poids_list.append(poids)
                ax.plot(cg, poids, 'k_', markersize=10)  # '

                if i % 5 == 0 and i>0:
                  ax.text(cg, poids, str(int(fuel_lbs)), horizontalalignment='right', verticalalignment='center', size=13)

            ax.plot(cg_list, poids_list, lw=1, color='red')

    ax.text(190, 5000, 'Fuel en lbs', horizontalalignment='left', verticalalignment='center', size=20)
    ax.text(190, 5100, 'Nombre de paras', horizontalalignment='left', verticalalignment='center', size=20, color='red')

    ax.set_ylabel('Masse (lbs)', fontsize=20)
    ax.set_ylim(EW_LBS - 100, MTOW_LBS + 500)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=len(ax.get_xticks()) * 2))
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=len(ax.get_yticks()) * 2))
    ax.tick_params(axis='both', which='major', labelsize=14, width=2, length=6)

    # Create secondary y-axis (right) for kg
    ax_kg = ax.twinx()
    ax_kg.set_ylim((EW_LBS - 100) / LBS2KG ,  (MTOW_LBS + 500 )/ LBS2KG)
    ax_kg.set_ylabel('Masse (kg)', fontsize=20)
    ax_kg.yaxis.set_major_locator(ticker.MaxNLocator(nbins=len(ax_kg.get_yticks()) * 4))
    ax_kg.tick_params(axis='both', which='major', labelsize=14, width=2, length=6)
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)


    # Set primary x-axis (bottom) to show cg in inches
    ax.set_xlim(179, 206)
    ax.set_xlabel('Position du CG (inches)', fontsize=20)

    ax_mac = ax.twiny()
    lower_limit_inch, upper_limit_inch = ax.get_xlim()
    lower_limit_mac = cg_inches_to_mac_percent(179, DATUM_LINE, MAC)
    upper_limit_mac = cg_inches_to_mac_percent(206, DATUM_LINE, MAC)
    ax_mac.set_xlim(lower_limit_mac, upper_limit_mac)
    ax_mac.set_xlabel('Position du CG(% of MAC)', fontsize=20)
    ax_mac.xaxis.set_major_locator(ticker.MaxNLocator(nbins=len(ax_mac.get_xticks()) * 5))
    ax_mac.tick_params(axis='both', which='major', labelsize=14, width=2, length=6)




    # Your existing code to finish up the plotting...
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)


def plot_envelope_critical_points(ax, EW_LBS, EW_MOMENT, MTOW_LBS,
                                   POIDS_PILOTE_KG, POIDS_PARA_KG, zones,
                                   DATUM_LINE, MAC, target_cg_in=200.5,
                                   n_max=20):
    """Dessine l'enveloppe (centrogramme) avec les coordonnees des points
    critiques ET les trajectoires CG/masse pour chaque nombre de paras (0..n_max)
    et chaque valeur de fuel."""
    LBS2KG = 2.20462
    DENSITY_FUEL_LBS_PER_GAL = 6.7

    def cg_m2mac(cg_in):
        return (cg_in - DATUM_LINE) / MAC * 100

    ENVELOPE = [(179.6, EW_LBS),
                (179.6, 5500),
                (193.37, 8000),
                (199.15, MTOW_LBS),
                (204.35, MTOW_LBS),
                (204.35, EW_LBS)]

    poly = patches.Polygon(xy=ENVELOPE, fill=True, facecolor='#E8F4FD',
                           edgecolor='#1F77B4', linewidth=2.5, zorder=2)
    ax.add_patch(poly)

    # Trajectoires N paras / fuel (points clippes a MTOW)
    fuel_lbs_range = list(range(0, int(332 * DENSITY_FUEL_LBS_PER_GAL), 100)) \
                     + [int(332 * DENSITY_FUEL_LBS_PER_GAL)]
    for n in range(n_max + 1):
        if 1 <= n <= 3:
            continue
        positions = _get_para_positions_caravan(
            n, zones, EW_LBS, EW_MOMENT, MTOW_LBS,
            POIDS_PILOTE_KG, POIDS_PARA_KG, target_cg_in=target_cg_in)
        list_positions = [x for (x, _y) in positions]
        cg_list, mass_list, fuel_list = [], [], []
        for fuel_lbs in fuel_lbs_range:
            fuel_gal = fuel_lbs / DENSITY_FUEL_LBS_PER_GAL
            _, mass, cg = compute_moment(list_positions, EW_LBS, EW_MOMENT,
                                           POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_gal)
            if mass > MTOW_LBS:
                continue
            cg_list.append(cg)
            mass_list.append(mass)
            fuel_list.append(fuel_lbs)
        if not cg_list:
            continue
        ax.plot(cg_list, mass_list, lw=1.0, color='#D62728',
                alpha=0.75, zorder=3)
        for cg, m in zip(cg_list, mass_list):
            ax.plot(cg, m, 'k_', markersize=6, zorder=3.5)
        ax.text(cg_list[0], mass_list[0], f' {n}', color='#D62728',
                fontsize=11, fontweight='bold', ha='right', va='top', zorder=5)
        if n == n_max:
            for i, (cg, m, f_lbs) in enumerate(zip(cg_list, mass_list, fuel_list)):
                if i % 3 == 0 and i > 0:
                    ax.text(cg, m, f' {int(f_lbs)}', color='#404040',
                            fontsize=8, ha='left', va='center', zorder=5)

    # Legende
    ax.text(0.02, 0.98, 'Rouge: N paras\nNoir: fuel (lbs)',
            transform=ax.transAxes, ha='left', va='top', fontsize=11,
            bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.4'),
            zorder=6)

    # Points critiques numerotes
    for idx, (cg, mass) in enumerate(ENVELOPE, start=1):
        ax.plot(cg, mass, 'o', color='#1F77B4', markersize=10, zorder=4)
        ax.text(cg, mass, f' {idx}', color='#1F77B4', fontsize=14,
                fontweight='bold', ha='left', va='bottom', zorder=5)

    # Tableau recapitulatif des coordonnees en bas a droite
    table_lines = ['Points critiques du centrogramme :',
                   f'{"#":<3}{"CG (in)":>9}{"%MAC":>10}{"Masse (lbs)":>14}{"Masse (kg)":>14}']
    for idx, (cg, mass) in enumerate(ENVELOPE, start=1):
        table_lines.append(
            f'{idx:<3}{cg:>9.2f}{cg_m2mac(cg):>9.2f}%{mass:>14.0f}{mass/LBS2KG:>14.0f}')
    ax.text(0.98, 0.02, '\n'.join(table_lines),
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=12, family='monospace',
            bbox=dict(facecolor='white', edgecolor='black',
                      boxstyle='round,pad=0.6'),
            zorder=6)

    ax.set_xlabel('Position CG (in)', fontsize=16)
    ax.set_ylabel('Masse (lbs)', fontsize=16)
    ax.set_xlim(175, 210)
    ax.set_ylim(EW_LBS - 200, MTOW_LBS + 200)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.tick_params(axis='both', which='major', labelsize=12)

    # Axe secondaire : %MAC en haut
    ax_mac = ax.twiny()
    ax_mac.set_xlim(cg_m2mac(175), cg_m2mac(210))
    ax_mac.set_xlabel('Position CG (% MAC)', fontsize=16)
    ax_mac.tick_params(axis='both', which='major', labelsize=12)

    # Axe secondaire : kg a droite
    ax_kg = ax.twinx()
    ax_kg.set_ylim((EW_LBS - 200) / LBS2KG, (MTOW_LBS + 200) / LBS2KG)
    ax_kg.set_ylabel('Masse (kg)', fontsize=16)
    ax_kg.tick_params(axis='both', which='major', labelsize=12)


In [4]:
def plot_data_table(ax, MTOW_LBS, DATUM_LINE, MAC, EW_LBS, EW_MOMENT, POIDS_PILOTE_KG, POIDS_PARA_KG, zones, target_cg_in=200.5):
    LBS2KG = 2.20462
    DENSITY_FUEL_LBS_PER_GAL = 6.7
    from matplotlib.patches import Polygon as _Poly
    from matplotlib.lines import Line2D

    def cg_inches_to_mac_percent(cg_inches):
        return (cg_inches - DATUM_LINE) / MAC * 100

    ax.axis('off')

    para_range = [0] + list(range(4, 21))
    fuel_range = list(range(320, 1761, 120))
    CG_LIMIT = 40.33
    MTOW_KG_VAL = MTOW_LBS / LBS2KG
    fuel_label = "Total\nfuel lbs"

    table_data = [[""] + [f"{x}" for x in fuel_range]]
    for para in para_range:
        row = [f"{para} Paras"]
        for fuel_lbs in fuel_range:
            fuel = fuel_lbs / DENSITY_FUEL_LBS_PER_GAL
            list_positions = [x for (x, _y) in _get_para_positions_caravan(para, zones, EW_LBS, EW_MOMENT, MTOW_LBS, POIDS_PILOTE_KG, POIDS_PARA_KG, target_cg_in=target_cg_in)]
            moment, total_mass_lbs, cg_in_inches = compute_moment(
                list_positions, EW_LBS, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel)
            cg_percent_mac = cg_inches_to_mac_percent(cg_in_inches)
            total_mass_kg = total_mass_lbs / LBS2KG
            row.append(f"{cg_percent_mac:.1f}% \n {total_mass_kg:.0f} kg")
        table_data.append(row)

    charge_row = ["Charge utile\nmax (kg)"]
    for fuel_lbs in fuel_range:
        fuel_gal = fuel_lbs / DENSITY_FUEL_LBS_PER_GAL
        _, mass0_lbs, _ = compute_moment([], EW_LBS, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_gal)
        charge_utile_kg = (MTOW_LBS - mass0_lbs) / LBS2KG
        charge_row.append(f"{charge_utile_kg:.0f}")
    table_data.append(charge_row)

    ncols = len(table_data[0])
    nrows = len(table_data)
    nb_data_rows = len(para_range)

    the_table = ax.table(cellText=table_data, loc='center', cellLoc='center',
                         bbox=[0, 0, 1, 1])
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(20)
    the_table.set_zorder(5)

    cells = the_table.get_celld()

    for j in range(1, ncols):
        cells[(0, j)].set_facecolor('#E0E0E0')
        cells[(0, j)].get_text().set_fontweight('bold')
        cells[(0, j)].get_text().set_fontsize(19)

    cell_split = cells[(0, 0)]
    cell_split.set_facecolor('none')
    cell_split.set_edgecolor('black')
    cell_split.set_linewidth(1.0)
    cell_split.get_text().set_text('')

    col_w = 1.0 / ncols
    row_h = 1.0 / nrows
    x_left  = 0.0
    x_right = 1 * col_w
    y_bot   = 1.0 - 1 * row_h
    y_top   = 1.0

    tri_gray = _Poly(
        [[x_left, y_top], [x_right, y_top], [x_right, y_bot]],
        facecolor='#E0E0E0', edgecolor='none',
        transform=ax.transAxes, zorder=3, clip_on=False)
    tri_blue = _Poly(
        [[x_left, y_top], [x_left, y_bot], [x_right, y_bot]],
        facecolor='#B3D9FF', edgecolor='none',
        transform=ax.transAxes, zorder=3, clip_on=False)
    ax.add_patch(tri_gray)
    ax.add_patch(tri_blue)

    ax.add_artist(Line2D(
        [x_left, x_right], [y_top, y_bot],
        transform=ax.transAxes, color='black', linewidth=1.5,
        zorder=4, clip_on=False))

    ax.text(x_left + 0.72 * col_w, y_bot + 0.75 * row_h,
            fuel_label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=12, fontweight='bold', zorder=5)
    ax.text(x_left + 0.22 * col_w, y_bot + 0.25 * row_h,
            "Nb\nparas",
            transform=ax.transAxes, ha='center', va='center',
            fontsize=12, fontweight='bold', zorder=5)

    is_red = [[False] * ncols for _ in range(nrows)]
    # Coloration par demi-cellule : haut = CG, bas = masse. Seule la partie
    # hors limite est coloree en rouge.
    import matplotlib.patches as _patches_dt
    for i in range(1, 1 + nb_data_rows):
        for j in range(1, ncols):
            cell_text = table_data[i][j]
            cells[(i, j)].get_text().set_fontsize(19)
            try:
                cg_pct, mass_kg = [float(v.strip('% kg')) for v in cell_text.split('\n')]
            except Exception:
                continue
            cg_red = cg_pct > CG_LIMIT
            mass_red = mass_kg > MTOW_KG_VAL
            if cg_red or mass_red:
                is_red[i][j] = True
            if not (cg_red or mass_red):
                continue
            # On retire le fond de la cellule et on dessine des demi-cellules
            cells[(i, j)].set_facecolor('none')
            x0 = j * col_w
            y0 = 1.0 - (i + 1) * row_h
            if cg_red:
                ax.add_patch(_patches_dt.Rectangle(
                    (x0, y0 + row_h / 2), col_w, row_h / 2,
                    facecolor='#F5B7B1', edgecolor='none',
                    transform=ax.transAxes, zorder=1, clip_on=False))
            if mass_red:
                ax.add_patch(_patches_dt.Rectangle(
                    (x0, y0), col_w, row_h / 2,
                    facecolor='#F5B7B1', edgecolor='none',
                    transform=ax.transAxes, zorder=1, clip_on=False))

    for i in range(1, 1 + nb_data_rows):
        cells[(i, 0)].set_facecolor('#E4EBF5')
        cells[(i, 0)].get_text().set_fontweight('bold')
        cells[(i, 0)].get_text().set_fontsize(18)

    # Ligne "0 paras" (i=1) : mise en valeur
    for j in range(ncols):
        c0 = cells[(1, j)]
        if j == 0:
            c0.set_facecolor('#FFE9A8')
        else:
            if not is_red[1][j]:
                c0.set_facecolor('#FFF6D5')
        c0.get_text().set_fontweight('bold')

    y_row0_top = 1.0 - 1 * row_h
    y_row0_bot = 1.0 - 2 * row_h
    ax.add_artist(Line2D([0, 1], [y_row0_top, y_row0_top],
                         transform=ax.transAxes, color='#B8860B', linewidth=2.0,
                         zorder=6, clip_on=False))
    ax.add_artist(Line2D([0, 1], [y_row0_bot, y_row0_bot],
                         transform=ax.transAxes, color='#B8860B', linewidth=2.0,
                         zorder=6, clip_on=False))

    # Ligne delim safe/rouge (escalier) : noir epais
    first_red = [ncols] * nrows
    for i in range(1, 1 + nb_data_rows):
        for j in range(1, ncols):
            if is_red[i][j]:
                first_red[i] = j
                break

    # Trace: verticaux aux lignes avec frontiere + horizontaux aux transitions.
    # On traite "au-dessus de la 1ere ligne de donnees" comme fc = ncols (tout safe).
    line_segments = []
    for i in range(1, 1 + nb_data_rows):
        fc = first_red[i]
        if 1 < fc < ncols:
            y_t = 1.0 - i * row_h
            y_b = 1.0 - (i + 1) * row_h
            line_segments.append([(fc * col_w, y_t), (fc * col_w, y_b)])

    prev_fc = ncols
    for i in range(1, 1 + nb_data_rows):
        fc = first_red[i]
        y_t = 1.0 - i * row_h
        if fc != prev_fc:
            x1 = min(prev_fc, fc) * col_w
            x2 = max(prev_fc, fc) * col_w
            line_segments.append([(x1, y_t), (x2, y_t)])
        prev_fc = fc

    for seg in line_segments:
        xs = [p[0] for p in seg]
        ys = [p[1] for p in seg]
        ax.add_artist(Line2D(xs, ys, transform=ax.transAxes,
                             color='white', linewidth=8.0, zorder=50,
                             clip_on=False, solid_joinstyle='miter',
                             solid_capstyle='butt'))
        ax.add_artist(Line2D(xs, ys, transform=ax.transAxes,
                             color='black', linewidth=5.0, zorder=51,
                             clip_on=False, solid_joinstyle='miter',
                             solid_capstyle='butt'))

    charge_row_idx = nrows - 1
    for j in range(ncols):
        cell = cells[(charge_row_idx, j)]
        cell.set_facecolor('#D5F5E3')
        cell.get_text().set_fontweight('bold')
        cell.get_text().set_fontsize(19 if j > 0 else 12)
        cell.set_edgecolor('black')


In [5]:
def _build_rotations_data(MTOW_LBS, DATUM_LINE, MAC, EW_LBS, EW_MOMENT,
                           POIDS_PARA_KG, POIDS_PILOTE_KG, zones,
                           target_cg_in=200.5):
    """Calcule les donnees communes aux deux tableaux rotations."""
    DENSITY_FUEL_LBS_PER_GAL = 6.7
    FUEL_PER_ROTATION = 120
    FUEL_RESERVE = 200
    LBS2KG = 2.20462

    fuel_range_list = list(range(320, 1761, 120))
    max_rotations = int(max((f - FUEL_RESERVE) // FUEL_PER_ROTATION for f in fuel_range_list))

    def max_paras_for_fuel(fuel_lbs):
        fuel_gal = fuel_lbs / DENSITY_FUEL_LBS_PER_GAL
        for n in range(20, -1, -1):
            list_positions = [x for (x, _y) in _get_para_positions_caravan(
                n, zones, EW_LBS, EW_MOMENT, MTOW_LBS,
                POIDS_PILOTE_KG, POIDS_PARA_KG, target_cg_in=target_cg_in)]
            _, total_mass_lbs, cg_in_inches = compute_moment(
                list_positions, EW_LBS, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_gal)
            if total_mass_lbs <= MTOW_LBS and (cg_in_inches - DATUM_LINE) / MAC * 100 <= 40.33:
                return n
        return 0

    all_rows = []
    for fuel in fuel_range_list:
        nb_rot = int((fuel - FUEL_RESERVE) // FUEL_PER_ROTATION)
        row_paras, row_charge, total_paras, total_charge = [], [], 0, 0
        for i in range(max_rotations):
            if i < nb_rot:
                fuel_at_rot = fuel - i * FUEL_PER_ROTATION
                n = max_paras_for_fuel(fuel_at_rot)
                _, mass0, _ = compute_moment(
                    [], EW_LBS, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG,
                    fuel_at_rot / DENSITY_FUEL_LBS_PER_GAL)
                charge_kg = round((MTOW_LBS - mass0) / LBS2KG)
                total_paras += n
                total_charge += charge_kg
                row_paras.append(str(n))
                row_charge.append(str(charge_kg))
            else:
                row_paras.append(None)
                row_charge.append(None)
        all_rows.append((fuel, nb_rot, row_paras, row_charge, total_paras, total_charge))

    return all_rows, max_rotations


def _draw_triangular_table(ax, table_data, header_bg, alt_color, color_total, subtitle, fs=12, has_total=True):
    """
    Tableau triangulaire : cellules None invisibles + barre verticale de fermeture
    a droite de la derniere cellule reelle (avant la colonne Total).
    """
    from matplotlib.patches import Rectangle as _Rect
    from matplotlib.lines import Line2D

    display_data = [['' if v is None else str(v) for v in row] for row in table_data]
    ncols = len(table_data[0])
    nrows = len(table_data)

    ax.axis('off')
    the_table = ax.table(cellText=display_data, loc='center', cellLoc='center',
                         bbox=[0, 0, 1, 1])
    the_table.auto_set_font_size(False)
    the_table.set_zorder(2)

    # last_real_col = derniere colonne non-None, hors colonne Total (ncols-1)
    last_real_col = {}
    for row in range(1, nrows):
        last_col = 0
        for col in range(ncols - 1 if has_total else ncols):
            if table_data[row][col] is not None:
                last_col = col
        last_real_col[row] = last_col

    cells = the_table.get_celld()

    for (row, col), cell in cells.items():
        if row >= nrows or col >= ncols:
            cell.set_visible(False)
            continue

        val = table_data[row][col]

        if val is None:
            cell.set_facecolor('white')
            cell.set_edgecolor('white')
            cell.get_text().set_text('')
            continue

        cell.set_linewidth(0.6)
        cell.set_edgecolor('#AAAAAA')

        if row == 0:
            cell.set_facecolor(header_bg)
            cell.get_text().set_color('white')
            cell.get_text().set_fontweight('bold')
            cell.get_text().set_fontsize(fs)
            cell.get_text().set_va('center')
        elif has_total and col == ncols - 1:
            cell.set_facecolor(color_total)
            cell.get_text().set_fontweight('bold')
            cell.get_text().set_fontsize(fs + 2)
        elif col <= 1:
            cell.set_facecolor('#E4EBF5')
            cell.get_text().set_fontweight('bold')
            cell.get_text().set_fontsize(fs + 1)
        else:
            cell.set_facecolor('#FFFFFF' if row % 2 == 0 else alt_color)
            cell.get_text().set_fontsize(fs + 1)

    # Barres de fermeture : verticale a droite de la derniere cellule reelle
    col_w = 1.0 / ncols
    row_h = 1.0 / nrows
    max_real_col = ncols - 2 if has_total else ncols - 1
    for row_idx in range(1, nrows):
        lv = last_real_col[row_idx]
        if lv < max_real_col:
            x_close = (lv + 1) * col_w
            y_bot = 1.0 - (row_idx + 1) * row_h
            y_top = 1.0 - row_idx * row_h

            # Barre verticale a droite de la derniere cellule reelle
            ax.add_artist(Line2D([x_close, x_close], [y_bot, y_top],
                                 transform=ax.transAxes,
                                 color='#AAAAAA', linewidth=0.6,
                                 zorder=100, clip_on=False,
                                 solid_capstyle='butt'))

            # Bas de la derniere cellule reelle (trait horizontal leger)
            x_left_cell = lv * col_w
            ax.add_artist(Line2D([x_left_cell, x_close], [y_bot, y_bot],
                                 transform=ax.transAxes,
                                 color='#AAAAAA', linewidth=0.6,
                                 zorder=100, clip_on=False,
                                 solid_capstyle='butt'))

    ax.set_title(subtitle, fontsize=fs + 2, pad=8, fontweight='bold')


def plot_rotations_page(ax_top, ax_bot, all_rows, max_rotations, title_text, POIDS_PARA_KG, fs=12):
    """Genere les deux tableaux rotations (paras + masse kg) sur une meme page."""
    rot_headers = [f'Rot.{i+1}' for i in range(max_rotations)]

    # --- Table 1 : nombre de paras ---
    header_p = ['Carburant\n(Lbs)', 'Nb\nrot.'] + rot_headers + ['Total\nparas']
    table_p = [header_p]
    for fuel, nb_rot, row_paras, _, total_paras, _ in all_rows:
        table_p.append([str(fuel), str(nb_rot)] + row_paras + [str(total_paras)])

    _draw_triangular_table(
        ax_top, table_p,
        header_bg='#2C5F8A', alt_color='#EAF2FB', color_total='#FFF3CD',
        subtitle=(f'{title_text}\n'
                  'Nombre de paras par rotation vs carburant embarque  '
                  '--  Conso: 120 lbs/rot au FL120  --  Reserve: 200 lbs (jour)'),
        fs=fs, has_total=True)

    # --- Table 2 : masse max embarquable (kg) = MTOW - EW - pilote - fuel ---
    header_c = ['Carburant\n(Lbs)', 'Nb\nrot.'] + rot_headers + ['Total\ncharge\n(kg)']
    table_c = [header_c]
    for fuel, nb_rot, _, row_charge, _, total_charge in all_rows:
        table_c.append([str(fuel), str(nb_rot)] + row_charge + [str(total_charge)])

    _draw_triangular_table(
        ax_bot, table_c,
        header_bg='#1E6B3C', alt_color='#EAFAF1', color_total='#D5F5E3',
        subtitle='Masse max embarquable (kg) par rotation  --  MTOW - EW - pilote - carburant',
        fs=fs, has_total=True)


In [6]:
def get_slots_la():

    zones = [
        {"label": "ZONE 0", "x0": 118., "x1": 155.4, "height": 53., "arm": 135.5},
        {"label": "ZONE 1", "x0": 155.4, "x1": 188.7, "height": 62., "arm": 172.1},
        {"label": "ZONE 2", "x0": 188.7, "x1": 246.8, "height": 62., "arm": 217.8},
        {"label": "ZONE 3", "x0": 246.8, "x1": 282.0, "height": 64., "arm": 264.4},
        {"label": "ZONE 4", "x0": 282.0, "x1": 307., "height": 57., "arm": 294.5},
        {"label": "ZONE 5", "x0": 307.0, "x1": 332., "height": 53., "arm": 319.5},
        {"label": "ZONE 6", "x0": 332., "x1": 356.0, "height": 46., "arm": 344},
        {"label": "ZONE -1", "x0": 100., "x1": 118., "height": 53., "arm": 109},
    ]

        ## TOP
    x0_top = zones[0]['x0'] + (zones[0]['x1'] - zones[0]['x0'])/2
    x1_top = zones[4]['x1'] 

    nb_slots_top = 8
    x_slots_top = [int(x0_top + i * (x1_top - x0_top) / (nb_slots_top - 1)) + 3 for i in range(nb_slots_top)]

    ## MIDDLE
    x0_middle = (zones[0]['x1']-zones[0]['x0']) * 0.85 + zones[0]['x0']
    x1_middle = (zones[4]['x1']-zones[4]['x0']) / 2 + zones[4]['x0']

    nb_slots_middle = 5
    x_slots_middle = [int(x0_middle + i * (x1_middle - x0_middle) / (nb_slots_middle - 1)) + 3 for i in range(nb_slots_middle)]

    ## BOTTOM
    x0_bot = zones[1]['x0'] + (zones[1]['x1'] - zones[1]['x0']) / 5
    x1_bot = zones[4]['x1'] 

    nb_slots_bot = 7
    x_slots_bot = [int(x0_bot + i * (x1_bot - x0_bot) / (nb_slots_bot - 1)) + 3 for i in range(nb_slots_bot)]

    return x_slots_top, x_slots_middle, x_slots_bot, zones

In [7]:
filling_order_bon = [16, 12, 7, 4, 1, 8, 13, 20] + \
                        [14, 8, 3, 9, 15] + \
                        [10, 6, 5, 2, 11, 17, 19]
list(sorted(filling_order_bon))

[1, 2, 3, 4, 5, 6, 7, 8, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20]

In [8]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from datetime import date
import os

EW_LBS = 4986
EW_MOMENT = 930.11184
POIDS_PILOTE_KG = 80
POIDS_PARA_KG = 90
MTOW_LBS = 9062
DATUM_LINE = 177.57
MAC = 66.40
IMMAT = 'C208B-B'
VERSION = date.today().strftime('%d/%m/%Y')
CIVIL = False  # bascule a True dans les jumeaux civils (gen_civil_notebooks.py)
TARGET_CG_IN = 200.5  # Milieu de la zone seulement limitee par la MTOW (199.15-204.35 in)

_x_slots_top, _x_slots_middle, _x_slots_bot, zones = get_slots_la()

all_rows, max_rotations = _build_rotations_data(
    MTOW_LBS, DATUM_LINE, MAC, EW_LBS, EW_MOMENT,
    POIDS_PARA_KG, POIDS_PILOTE_KG, zones,
    target_cg_in=TARGET_CG_IN)

version_fn = VERSION.replace('/', '-')
OUTPUT_DIR = os.path.join(os.pardir, 'output')  # notebooks/ -> output/
os.makedirs(OUTPUT_DIR, exist_ok=True)
filename = os.path.join(
    OUTPUT_DIR, f'Planches_centrage_{IMMAT}_90kg_{version_fn}.pdf')

with PdfPages(filename) as pdf:
    title_text = f'{IMMAT}: poids para: {POIDS_PARA_KG} (kg), poids pilote: {POIDS_PILOTE_KG} (kg)'

    # Page 1a/1b - Configuration : 2 pages de 9 mini-plans (3x3)
    n_values_all = [0] + list(range(4, 21))
    split = len(n_values_all) // 2 + (len(n_values_all) % 2)
    page_groups = [n_values_all[:split], n_values_all[split:]]
    for page_idx, n_vals in enumerate(page_groups):
        fig1, axes1 = plt.subplots(3, 3, figsize=(11.7*2, 8.3*2), dpi=200)
        suffix = f" (page {page_idx+1}/2)"
        fig1.suptitle(title_text + " - Placement des paras selon le nombre a bord" + suffix + "\n"
                      f"EW: {EW_LBS} lbs  |  EW Moment: {EW_MOMENT} lbs.in  |  MTOW: {MTOW_LBS} lbs  |  Version: {VERSION}",
                      fontsize=20)
        plot_conf_grid(axes1, zones, 20, EW_LBS, EW_MOMENT, MTOW_LBS,
                       POIDS_PILOTE_KG, POIDS_PARA_KG,
                       n_values=n_vals, target_cg_in=TARGET_CG_IN)
        fig1.tight_layout(rect=[0, 0, 1, 0.94])
        pdf.savefig(fig1, bbox_inches=_draw_profile_frame(fig1, CIVIL))
        plt.close(fig1)

    # Page 2 - Tableau masse et centrage
    fig4 = plt.figure(figsize=(11.7*2, 8.3*2), dpi=200)
    plt.title(title_text + " - Tableau masse & centrage", fontsize=30, pad=20)
    ax4 = fig4.gca()
    plot_data_table(ax4, MTOW_LBS, DATUM_LINE, MAC, EW_LBS, EW_MOMENT,
                    POIDS_PILOTE_KG, POIDS_PARA_KG, zones,
                    target_cg_in=TARGET_CG_IN)
    pdf.savefig(fig4, bbox_inches=_draw_profile_frame(fig4, CIVIL))
    plt.close(fig4)

    # Page 3 - Rotations
    fig5, (ax5_top, ax5_bot) = plt.subplots(
        2, 1, figsize=(11.7*2, 8.3*2), dpi=200,
        gridspec_kw={'hspace': 0.18, 'top': 0.97, 'bottom': 0.02})
    fig5.patch.set_facecolor('#FAFAFA')
    plot_rotations_page(ax5_top, ax5_bot, all_rows, max_rotations, title_text, POIDS_PARA_KG, fs=12)
    pdf.savefig(fig5, bbox_inches=_draw_profile_frame(fig5, CIVIL))
    plt.close(fig5)


    # Annexe - Centrogramme (enveloppe + points critiques)
    fig_cp = plt.figure(figsize=(11.7*2, 8.3*2), dpi=200)
    ax_cp = fig_cp.gca()
    plot_envelope_critical_points(ax_cp, EW_LBS, EW_MOMENT, MTOW_LBS,
                                   POIDS_PILOTE_KG, POIDS_PARA_KG, zones,
                                   DATUM_LINE, MAC,
                                   target_cg_in=TARGET_CG_IN, n_max=20)
    fig_cp.suptitle(title_text + " - Annexe : centrogramme & points critiques", fontsize=22)
    fig_cp.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig_cp, bbox_inches=_draw_profile_frame(fig_cp, CIVIL))
    plt.close(fig_cp)

print(f"PDF genere : {filename} (5 pages)")


PDF genere : ../output/Planches_centrage_C208B-B_90kg_02-09-2026.pdf (5 pages)
